In [1]:
!pip install -U torchao

In [2]:
!pip install -q transformers peft datasets accelerate librosa soundfile \
  noisereduce jiwer evaluate gradio torchaudio omegaconf wandb \
  praat-parselmouth seaborn matplotlib

In [3]:
from datasets import load_dataset

ds = load_dataset("hf-internal-testing/librispeech_asr_demo", "clean", split="validation")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
import transformers, peft, torch

print(transformers.__version__)
print(peft.__version__)
print(torch.__version__)

4.41.2
0.11.1
2.11.0+cu128


In [5]:
!pip uninstall -y transformers peft accelerate
!pip install transformers==4.41.2
!pip install peft==0.11.1
!pip install accelerate==0.30.1

Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2
Found existing installation: peft 0.11.1
Uninstalling peft-0.11.1:
  Successfully uninstalled peft-0.11.1
Found existing installation: accelerate 0.30.1
Uninstalling accelerate-0.30.1:
  Successfully uninstalled accelerate-0.30.1
  Using cached transformers-4.41.2-py3-none-any.whl.metadata (43 kB)
Using cached transformers-4.41.2-py3-none-any.whl (9.1 MB)


  Using cached peft-0.11.1-py3-none-any.whl.metadata (13 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
Using cached peft-0.11.1-py3-none-any.whl (251 kB)
Using cached accelerate-1.14.0-py3-none-any.whl (389 kB)


  Using cached accelerate-0.30.1-py3-none-any.whl.metadata (18 kB)
Using cached accelerate-0.30.1-py3-none-any.whl (302 kB)
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0


In [7]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-small"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=ds_prepared,
    data_collator=data_collator,
)

trainer.train()

Step,Training Loss
10,3.141300
20,0.990800
30,0.676600
40,0.429400
50,0.172200


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}


TrainOutput(global_step=57, training_loss=0.9562973981363732, metrics={'train_runtime': 81.6304, 'train_samples_per_second': 2.683, 'train_steps_per_second': 0.698, 'total_flos': 6.320020267008e+16, 'train_loss': 0.9562973981363732, 'epoch': 3.0})

In [11]:
import os

for root, dirs, files in os.walk("."):
    if "config.json" in files:
        print(root)

./checkpoints/lora_run1/checkpoint-50


In [13]:
from transformers import WhisperForConditionalGeneration

ft_model = WhisperForConditionalGeneration.from_pretrained(
    "./checkpoints/lora_run1/checkpoint-50"   # replace with actual path
).to("cuda")

In [15]:
generated_ids = baseline_model.generate(
    input_features,
    language="en",
    task="transcribe"
)

In [16]:
import os
print(os.listdir("."))
print(os.listdir("./checkpoints"))

['.config', 'checkpoints', 'sample_data']
['lora_run1']


In [17]:
import torch
import jiwer
from transformers import WhisperForConditionalGeneration

# Load fine-tuned model
ft_model = WhisperForConditionalGeneration.from_pretrained(
    "./checkpoints/lora_run1/checkpoint-50"
).to("cuda")

ft_model.eval()

refs = [x["text"] for x in ds.select(range(20))]
preds = []

for item in ds.select(range(20)):
    audio = item["audio"]["array"]
    sr = item["audio"]["sampling_rate"]

    inputs = processor(
        audio,
        sampling_rate=sr,
        return_tensors="pt"
    ).input_features.to("cuda")

    with torch.no_grad():
        ids = ft_model.generate(
            inputs,
            language="en",
            task="transcribe"
        )

    text = processor.batch_decode(
        ids,
        skip_special_tokens=True
    )[0]

    preds.append(text)

wer = jiwer.wer(refs, preds)

print("WER:", round(wer * 100, 2), "%")

WER: 0.45 %


In [18]:
import gradio as gr

def transcribe_fn(audio_file, ref_text):
    import soundfile as sf
    audio, sr = sf.read(audio_file)
    inputs = processor(audio, sampling_rate=sr, return_tensors="pt").input_features.to("cuda")
    with torch.no_grad():
        gen = ft_model.generate(inputs, language="en", task="transcribe")
    transcript = processor.tokenizer.decode(gen[0], skip_special_tokens=True)
    wer_str = ""
    if ref_text.strip():
        wer_str = f"WER: {jiwer.wer(ref_text, transcript)*100:.2f}%"
    return transcript, wer_str

demo = gr.Interface(
    fn=transcribe_fn,
    inputs=[gr.Audio(type="filepath", label="Upload song clip"),
            gr.Textbox(label="Reference lyrics (optional)")],
    outputs=[gr.Textbox(label="Transcribed Lyrics"), gr.Textbox(label="WER")],
    title="🎤 AutoLyrics — Singing Voice Transcription"
)
demo.launch(share=True)  # gives you a public link!

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2f82d4a94563832e8e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
